<a href="https://colab.research.google.com/github/Iberasa3/Hollow-Hornet/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd

# El separador '\t' le dice a pandas que use el tabulador
df = pd.read_csv('avispas_avistamientos.csv', sep='\t') #Duh el separador era esta shit

# Ahora sí, vamos a comprobar que se han separado
print(f"Número de columnas: {df.shape[1]}")
print(f"Nombres de las columnas: {df.columns.tolist()[:5]}...")

Número de columnas: 50
Nombres de las columnas: ['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum']...


Efectivamente este es mi proyecto de avispas, de momento no tengo mucho más que añadir. Cuando termine de limpiar este dataset proyectaré las coordenadas en GEE

Un par de apuntes para mí mismo, probablemente acabemos considerando a todas las subespecies de avispa asiatica como una sola para no tener que liarnos entre las distintas subespecies.

Varios factores que tengo que tener en cuenta para poder filtrar bien los datos que se me están presentando:

1- Los datos anteriores a 2005, si los hay, no hay que tenerlos en cuenta porque Vespa velutina entró a europa en 2004

2- Cuidado con null island (hecho)

3- cuidado con los rangos en los que el avistamiento tiene gran índice de error (hecho)

4- cuidado con los avistamientos no-salvajes (hecho, preservados eliminados)

5- hay que chekiar las diferentes subespecies por si acaso

6- hay que chekiar solo los que sean de europa (hecho)


Tengo que leer algunos papers para que esto parezca más académico, pero tampoco pasarme 150 días leyendo papers para justificar toda la literatura


In [23]:
df['infraspecificEpithet'].unique() #Si la celda está vacía entonces asumimos que no pertenece a ninguna subespecie o que es 'común'

array([nan, 'nigrithorax', 'variana', 'auraria', 'flavitarsus',
       'velutina', 'karnyi', 'celebensis', 'ardens', 'sumbana',
       'divergens', 'floresiana', 'timorensis'], dtype=object)

In [24]:
paises_europa = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL'] #El análisis se basa en europa así que nos quedamos solo con los países europeos. La mayoría de registros de todas formas son europeos.
df_europa_0 = df[df['countryCode'].isin(paises_europa)]
print(f"Pasamos de {len(df)} a {len(df_europa_0)} registros.")

Pasamos de 377899 a 372895 registros.


In [25]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor
conteo_paises = df_europa_0['countryCode'].value_counts()

print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

--- Avistamientos por País en Europa ---
countryCode
BE    201699
NL     72038
FR     54466
CH     20092
DE     17921
ES      3518
PT      2251
LU       642
IT       211
IE        37
GB        20
Name: count, dtype: int64


In [26]:
#Quitamos todas las coordenadas basura y los nulls
df_coords0 = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df_coords0 = df_coords0.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Registros originales: {len(df)}")
print(f"Registros después de haber quitado los nulls y los null-islands: {len(df_coords0)}")

Registros originales: 377899
Registros después de haber quitado los nulls y los null-islands: 376580


In [27]:
#Quitamos las instancias en las que la incertidumbre es superior a 1km. Habría que ver por qué coño existen incertidumbres tan grandes
umbral_maximo = 1000
df_fin = df_coords0[(df_coords0['coordinateUncertaintyInMeters'] <= 1000) & (df['year'] >= 2005)]

Hay que tener cuidado con las instancias que se hayan tomado en el mismo instante, puede ser que un pibe haya apuntado varias veces a una misma colmena o incluso a una misma avispa, y que por ello haya zonas sobrerepresentadas.
Ahora habría que ver cómo chota hacemos esto del spatial thinning.
Como hemos descargado la versión simple del Darwin Core igual tenemos datos que nos ayudan con esto. De hecho igual en un futuro lo hago directamente con DWC

In [28]:
df_fin = df_fin[df_fin['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df_fin['basisOfRecord'].unique())

['HUMAN_OBSERVATION']


Vamos con el spatial thinning.

Podemos aplicar una agrupación por celda, útil para evitar sobremuestreos pero falla para detectar zonas más idóneas en las que las avispas hayan podido construir sus colmenas. Luego podemos sumarlo a un check de correlación para ver si nos ha salido bien.

La cosa es que si vamos a aplicar un random forest se asume que cada dato va a ser independiente, el modelo no va a saber que los distintos puntos que están cerca entre sí pueden reflejar una colmena próxima. Además al reflejar los puntos en el propio mapa la resolución no va a dar para más, así que creo que hacer el spatial thinning sin tener en cuenta la posibilidad de colmena es, de momento, suficiente. Esto es un modelo de distribución, no uno de abundancia.

¿Cuánto vuela una avispa desde su nido?
Por lo que he leído en internet de fuentes fiables, todas coinciden en que rara vez sobrepasan 1km o el kilometro y medio desde sus nidos, así que si hay algún avistamiento de avispa debe de haber un nido relativamente cerca casi seguro



In [29]:
num_duplicados = df_fin.duplicated().sum()
print(f"Registros con el mismo gbifID: {num_duplicados}")
df.drop_duplicates(subset='gbifID', inplace=True)

Registros con el mismo gbifID: 0


Hemos comprobado también que no haya avistamientos con un ID repetido. Se asume que los estándares de Darwin Core no permiten duplicados en sus datasets, pero por si acaso

In [30]:
# 0.015 grados de latitud son aproximadamente 1.1 km, así que cada celda va a tener aprox 1,1km x 1,1km
# Es una resolución estándar para modelos climáticos de 1km.
res = 0.015

# Dividimos por la resolución, redondeamos al entero más cercano y volvemos a multiplicar.
df['lat_grid'] = (df['decimalLatitude'] / res).round() * res
df['lon_grid'] = (df['decimalLongitude'] / res).round() * res

df_thinned = df.drop_duplicates(subset=['lat_grid', 'lon_grid'], keep='first').copy()

print(f"Dataset original: {len(df)} registros")
print(f"Dataset tras thinning (1km): {len(df_thinned)} registros")
print(f"Registros eliminados: {len(df) - len(df_thinned)}")

df_thinned[['decimalLatitude', 'lat_grid', 'decimalLongitude', 'lon_grid']].head()

Dataset original: 377899 registros
Dataset tras thinning (1km): 45402 registros
Registros eliminados: 332497


,decimalLatitude,lat_grid,decimalLongitude,lon_grid
0,46.56820,46.575,3.32410,3.330
1,44.79444,44.790,-0.54324,-0.540
2,51.11106,51.105,4.07784,4.080
3,51.21826,51.225,2.90049,2.895
4,50.86031,50.865,4.62720,4.620


Esto de arriba está full hecho con IA, tengo que revisarlo y hacer los cambios pertinentes.

In [33]:
# Contamos cuántos registros hay por país y lo ordenamos de mayor a menor
conteo_paises = df_thinned['countryCode'].value_counts()

print("--- Avistamientos por País en Europa ---")
print(conteo_paises)

--- Avistamientos por País en Europa ---
countryCode
FR    19189
BE    10584
DE     7657
ES     1776
NL     1549
PT     1126
CN      878
CH      672
KR      436
LU      256
HK      219
TW      215
ID      192
IN      144
IT      137
VN       83
PK       48
NP       41
TH       39
JE       21
AT       18
SE       14
BT       12
LA       10
MO       10
TL        9
IE        9
US        9
GB        8
JP        7
MY        7
MM        5
AF        3
MC        3
GG        3
NZ        2
AL        2
NO        2
ZZ        1
DK        1
DZ        1
CZ        1
EH        1
GR        1
HU        1
Name: count, dtype: int64


In [31]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='vespa-489513')

puntos_geospatiales = geemap.pandas_to_ee(
    df_thinned,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)

print(f"Colección lista con {df_thinned.shape} puntos tras el thinning de 1.6km.")

Colección lista con (45402, 52) puntos tras el thinning de 1.6km.


In [32]:
columnas_necesarias = ['decimalLatitude', 'decimalLongitude', 'year']
df_filtrado = df_thinned[columnas_necesarias].copy()

df_filtrado = df_filtrado.dropna(subset=['decimalLatitude', 'decimalLongitude'])

puntos_geospatiales = geemap.pandas_to_ee(
    df_filtrado,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)

Map = geemap.Map()
Map.addLayer(puntos_geospatiales, {'color': 'red'}, 'Vespa velutina (Optimized)')
Map.centerObject(puntos_geospatiales, 6)
Map

KeyboardInterrupt: 